In [ ]:
%pip install -q datasets pandas numpy matplotlib scikit-learn xgboost lightgbm scikit-uplift shap networkx joblib


In [ ]:
# -*- coding: utf-8 -*-
"""
CPDIF: COMPLETE END-TO-END EXPERIMENT
Dataset: Hugging Face criteo/criteo-uplift

This single script:
1. Downloads and samples the Criteo uplift dataset.
2. Creates one reproducible stratified train/validation/test split.
3. Benchmarks 12 uplift estimators and uses validation-selected CPDIF-S-LR as the single primary model.
4. Benchmarks 12 causal-uplift estimator configurations.
5. Estimates regression-adjusted and propensity-score-matched ATEs.
6. Runs covariate-dependent treated-arm subsampling experiments.
7. Produces counterfactual ITEs, response segments and targeting policies.
8. Evaluates policies using randomized-trial IPW estimates and the specified
   economic settings.
9. Runs threshold sensitivity and paired bootstrap uncertainty analyses.
10. Produces figures, CSV tables, LaTeX tables, fitted models and metadata.

IMPORTANT EXPERIMENTAL IMPLEMENTATION NOTES
- CPDIF-S-LR (S-learner with logistic regression) is the single primary CPDIF ITE and policy model.
- CPDIF-S (XGBoost) and the other configurations are benchmark comparators only.
- The additional learners are estimator comparisons within the ITE stage;
  they do not replace the CPDIF framework.
- SHAP explains the predictive outcome model associationally; it is not a
  causal explanation.
- The randomized assignment propensity is set to 0.85 in this configuration.
  Verify this against the dataset's experimental documentation before
  interpreting IPW/AIPW estimates.
- `visit` and `exposure` are excluded from baseline predictors.
- All model selection is based on validation Qini AUC. Test outcomes are
  reserved for final evaluation.
- The controlled-confounding experiment selectively retains treated records
  using the specified logistic covariate-dependent selection mechanism.
- Monetary ROI is evaluated using USD 50 per incremental conversion and
  USD 0.10 advertising cost per targeted user.
"""

import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import joblib

from datasets import load_dataset

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklift.metrics import uplift_at_k, qini_auc_score, uplift_auc_score

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# 1. EXPERIMENT CONFIGURATION
# ---------------------------------------------------------------------

SEED = 42
DATASET_ID = "criteo/criteo-uplift"
SAMPLE_FRACTION = 0.10

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

# Confirm against the dataset's randomization documentation.
PROPENSITY = 0.85

FEATURES = [f"f{i}" for i in range(12)]
TREATMENT_COL = "treatment"
OUTCOME_COL = "conversion"

# Main CPDIF decision configuration
POSITIVE_ITE_THRESHOLD = 0.001
NEGATIVE_ITE_THRESHOLD = -0.001
TARGET_FRACTION = 0.087

# Controlled-confounding configuration
SELECTION_COEFFICIENTS = (0.8, 0.5, -0.4)
TREATED_RETENTION_TARGET = 0.70
CONFOUNDING_STRENGTHS = (0.5, 1.0, 1.5)

# Economic configuration
CONVERSION_VALUE_USD = 50.0
AD_COST_PER_TARGETED_USER_USD = 0.10

# Threshold sensitivity: positive and negative thresholds are symmetric.
THRESHOLD_MAGNITUDES = (0.0005, 0.00075, 0.001, 0.002, 0.005)

# Bootstrap configuration
N_BOOTSTRAP = 1000
N_CV_FOLDS = 5
SHAP_SAMPLE_SIZE = 2500
PSM_CALIPER = 0.02

OUTPUT_DIR = Path("CPDIF_Results")
FIGURE_DIR = OUTPUT_DIR / "Figures"
TABLE_DIR = OUTPUT_DIR / "Tables"
MODEL_DIR = OUTPUT_DIR / "Models"

for folder in (OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)


# ---------------------------------------------------------------------
# 2. GENERAL UTILITIES
# ---------------------------------------------------------------------

def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"{filename}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{filename}.pdf", bbox_inches="tight")
    plt.close(fig)


def save_table(frame, filename):
    frame.to_csv(TABLE_DIR / f"{filename}.csv", index=False)
    with open(TABLE_DIR / f"{filename}.tex", "w", encoding="utf-8") as f:
        f.write(frame.to_latex(index=False, escape=True, float_format="%.6f"))


def clean_effect(values):
    return np.nan_to_num(
        np.asarray(values, dtype=float),
        nan=0.0, posinf=0.0, neginf=0.0
    )


def predict_probability_one(model, X):
    probabilities = model.predict_proba(X)
    classes = list(model.classes_)
    if 1 not in classes:
        return np.zeros(len(X), dtype=float)
    return probabilities[:, classes.index(1)]


def check_binary(name, values):
    unique = set(np.unique(values).tolist())
    if not unique.issubset({0, 1}):
        raise ValueError(f"{name} must contain only 0 and 1; found {unique}.")


# ---------------------------------------------------------------------
# 3. DATA LOADING AND REPRODUCIBLE SPLITTING
# ---------------------------------------------------------------------

def load_criteo_data():
    print(f"Loading dataset: {DATASET_ID}")
    dataset = load_dataset(DATASET_ID)

    if hasattr(dataset, "keys") and "train" in dataset:
        dataset = dataset["train"]
    elif hasattr(dataset, "keys") and not hasattr(dataset, "column_names"):
        dataset = dataset[list(dataset.keys())[0]]

    total_rows = len(dataset)
    sample_rows = max(1, int(total_rows * SAMPLE_FRACTION))
    print(f"Dataset rows: {total_rows:,}; sample rows: {sample_rows:,}")

    sampled = dataset.shuffle(seed=SEED).select(range(sample_rows))
    required = FEATURES + [TREATMENT_COL, OUTCOME_COL]
    missing = [column for column in required if column not in sampled.column_names]
    if missing:
        raise ValueError(f"Dataset is missing required columns: {missing}")

    frame = sampled.select_columns(required).to_pandas()
    del dataset, sampled
    gc.collect()

    for column in required:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame = frame.replace([np.inf, -np.inf], np.nan)
    frame = frame.dropna(subset=[TREATMENT_COL, OUTCOME_COL]).reset_index(drop=True)
    frame[TREATMENT_COL] = frame[TREATMENT_COL].astype(int)
    frame[OUTCOME_COL] = frame[OUTCOME_COL].astype(int)

    check_binary(TREATMENT_COL, frame[TREATMENT_COL].to_numpy())
    check_binary(OUTCOME_COL, frame[OUTCOME_COL].to_numpy())

    return frame


def create_data_splits(frame):
    X_raw = frame[FEATURES]
    T = frame[TREATMENT_COL].to_numpy(dtype=int)
    Y = frame[OUTCOME_COL].to_numpy(dtype=int)

    strata = T.astype(str) + "_" + Y.astype(str)
    indices = np.arange(len(frame))

    train_validation_idx, test_idx = train_test_split(
        indices,
        test_size=TEST_FRACTION,
        random_state=SEED,
        stratify=strata
    )

    validation_relative = VALIDATION_FRACTION / (
        TRAIN_FRACTION + VALIDATION_FRACTION
    )

    train_idx, validation_idx = train_test_split(
        train_validation_idx,
        test_size=validation_relative,
        random_state=SEED,
        stratify=strata[train_validation_idx]
    )

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_raw.iloc[train_idx]).astype(np.float32)
    X_validation = imputer.transform(X_raw.iloc[validation_idx]).astype(np.float32)
    X_test = imputer.transform(X_raw.iloc[test_idx]).astype(np.float32)

    data = {
        "X_train": X_train,
        "X_validation": X_validation,
        "X_test": X_test,
        "T_train": T[train_idx],
        "T_validation": T[validation_idx],
        "T_test": T[test_idx],
        "Y_train": Y[train_idx],
        "Y_validation": Y[validation_idx],
        "Y_test": Y[test_idx],
        "train_idx": train_idx,
        "validation_idx": validation_idx,
        "test_idx": test_idx,
        "imputer": imputer,
        "frame": frame
    }

    # Persist exact row indices so all subsequent analyses use this split.
    np.savez_compressed(
        OUTPUT_DIR / "Data_Split_Indices.npz",
        train_idx=train_idx,
        validation_idx=validation_idx,
        test_idx=test_idx
    )
    joblib.dump(imputer, MODEL_DIR / "Median_Imputer.joblib")
    return data


# ---------------------------------------------------------------------
# 4. MODEL FACTORIES
# ---------------------------------------------------------------------

def make_classifier(algorithm):
    if algorithm == "XGB":
        return XGBClassifier(
            n_estimators=400, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0,
            min_child_weight=5, objective="binary:logistic",
            eval_metric="logloss", random_state=SEED,
            n_jobs=-1, tree_method="hist"
        )

    if algorithm == "LGBM":
        return LGBMClassifier(
            n_estimators=400, num_leaves=31, learning_rate=0.05,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            reg_lambda=5.0, min_child_samples=30,
            verbosity=-1, random_state=SEED, n_jobs=-1
        )

    if algorithm == "ET":
        return ExtraTreesClassifier(
            n_estimators=300, max_features=0.8,
            min_samples_leaf=5, random_state=SEED, n_jobs=-1
        )

    if algorithm == "LR":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                C=1.0, max_iter=1000, solver="lbfgs"
            ))
        ])

    raise ValueError(f"Unknown classifier algorithm: {algorithm}")


def make_regressor(algorithm):
    if algorithm == "XGB":
        return XGBRegressor(
            n_estimators=400, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0,
            min_child_weight=5, objective="reg:squarederror",
            random_state=SEED, n_jobs=-1, tree_method="hist"
        )

    if algorithm == "LGBM":
        return LGBMRegressor(
            n_estimators=400, num_leaves=31, learning_rate=0.05,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            reg_lambda=5.0, min_child_samples=30,
            verbosity=-1, random_state=SEED, n_jobs=-1
        )

    if algorithm == "ET":
        return ExtraTreesRegressor(
            n_estimators=300, min_samples_leaf=5,
            random_state=SEED, n_jobs=-1
        )

    raise ValueError(f"Unknown regressor algorithm: {algorithm}")


# ---------------------------------------------------------------------
# 5. CAUSAL UPLIFT LEARNERS
# ---------------------------------------------------------------------

class SLearner:
    """Original CPDIF counterfactual outcome model f(X,T)."""
    def __init__(self, algorithm="XGB"):
        self.algorithm = algorithm
        self.model = make_classifier(algorithm)

    def fit(self, X, T, Y):
        self.model.fit(np.column_stack([X, T]), Y)
        return self

    def predict_uplift(self, X):
        X1 = np.column_stack([X, np.ones(len(X))])
        X0 = np.column_stack([X, np.zeros(len(X))])
        return clean_effect(
            predict_probability_one(self.model, X1)
            - predict_probability_one(self.model, X0)
        )

    def predict_counterfactuals(self, X):
        X1 = np.column_stack([X, np.ones(len(X))])
        X0 = np.column_stack([X, np.zeros(len(X))])
        p1 = predict_probability_one(self.model, X1)
        p0 = predict_probability_one(self.model, X0)
        return p0, p1, clean_effect(p1 - p0)


class TLearner:
    def __init__(self, algorithm):
        self.algorithm = algorithm
        self.control_model = make_classifier(algorithm)
        self.treated_model = make_classifier(algorithm)

    def fit(self, X, T, Y):
        control = T == 0
        treated = T == 1
        self.control_model.fit(X[control], Y[control])
        self.treated_model.fit(X[treated], Y[treated])
        return self

    def predict_uplift(self, X):
        return clean_effect(
            predict_probability_one(self.treated_model, X)
            - predict_probability_one(self.control_model, X)
        )


class XLearner:
    def __init__(self, algorithm, propensity):
        self.algorithm = algorithm
        self.propensity = float(propensity)
        self.control_model = make_classifier(algorithm)
        self.treated_model = make_classifier(algorithm)
        self.tau_control = make_regressor(algorithm)
        self.tau_treated = make_regressor(algorithm)

    def fit(self, X, T, Y):
        control = T == 0
        treated = T == 1
        self.control_model.fit(X[control], Y[control])
        self.treated_model.fit(X[treated], Y[treated])

        d_treated = Y[treated] - predict_probability_one(
            self.control_model, X[treated]
        )
        d_control = predict_probability_one(
            self.treated_model, X[control]
        ) - Y[control]

        self.tau_treated.fit(X[treated], d_treated)
        self.tau_control.fit(X[control], d_control)
        return self

    def predict_uplift(self, X):
        tau0 = self.tau_control.predict(X)
        tau1 = self.tau_treated.predict(X)
        return clean_effect(
            self.propensity * tau0 + (1.0 - self.propensity) * tau1
        )


class DRLearner:
    """Doubly robust learner with cross-fitted outcome nuisance models."""
    def __init__(self, algorithm, propensity, folds=N_CV_FOLDS):
        self.algorithm = algorithm
        self.propensity = float(propensity)
        self.folds = int(folds)
        self.effect_model = make_regressor(algorithm)

    def fit(self, X, T, Y):
        strata = T.astype(str) + "_" + Y.astype(str)
        cv = StratifiedKFold(
            n_splits=self.folds, shuffle=True, random_state=SEED
        )
        mu0 = np.zeros(len(Y), dtype=float)
        mu1 = np.zeros(len(Y), dtype=float)

        for fold, (fit_idx, hold_idx) in enumerate(cv.split(X, strata), 1):
            idx0 = fit_idx[T[fit_idx] == 0]
            idx1 = fit_idx[T[fit_idx] == 1]
            model0 = make_classifier(self.algorithm)
            model1 = make_classifier(self.algorithm)
            model0.fit(X[idx0], Y[idx0])
            model1.fit(X[idx1], Y[idx1])
            mu0[hold_idx] = predict_probability_one(model0, X[hold_idx])
            mu1[hold_idx] = predict_probability_one(model1, X[hold_idx])
            print(f"      DR cross-fitting fold {fold}/{self.folds}")

        e = self.propensity
        pseudo = (
            mu1 - mu0
            + T * (Y - mu1) / e
            - (1 - T) * (Y - mu0) / (1 - e)
        )
        self.effect_model.fit(X, pseudo)
        return self

    def predict_uplift(self, X):
        return clean_effect(self.effect_model.predict(X))


class RLearner:
    """R learner with cross-fitted marginal outcome nuisance model."""
    def __init__(self, algorithm, propensity, folds=N_CV_FOLDS):
        self.algorithm = algorithm
        self.propensity = float(propensity)
        self.folds = int(folds)
        self.effect_model = make_regressor(algorithm)

    def fit(self, X, T, Y):
        strata = T.astype(str) + "_" + Y.astype(str)
        cv = StratifiedKFold(
            n_splits=self.folds, shuffle=True, random_state=SEED
        )
        m_hat = np.zeros(len(Y), dtype=float)

        for fold, (fit_idx, hold_idx) in enumerate(cv.split(X, strata), 1):
            nuisance = make_classifier(self.algorithm)
            nuisance.fit(X[fit_idx], Y[fit_idx])
            m_hat[hold_idx] = predict_probability_one(
                nuisance, X[hold_idx]
            )
            print(f"      R cross-fitting fold {fold}/{self.folds}")

        residual_t = T.astype(float) - self.propensity
        residual_y = Y.astype(float) - m_hat
        valid = np.abs(residual_t) > 1e-8
        pseudo = residual_y[valid] / residual_t[valid]
        weights = residual_t[valid] ** 2
        self.effect_model.fit(X[valid], pseudo, sample_weight=weights)
        return self

    def predict_uplift(self, X):
        return clean_effect(self.effect_model.predict(X))


MODEL_REGISTRY = {
    "CPDIF-S": ("S", "XGB"),
    "CPDIF-S-LGBM": ("S", "LGBM"),
    "CPDIF-S-ET": ("S", "ET"),
    "CPDIF-S-LR": ("S", "LR"),
    "CPDIF-TXGB": ("T", "XGB"),
    "CPDIF-TLGBM": ("T", "LGBM"),
    "CPDIF-X": ("X", "XGB"),
    "CPDIF-X-LGBM": ("X", "LGBM"),
    "CPDIF-DR": ("DR", "XGB"),
    "CPDIF-DR-LGBM": ("DR", "LGBM"),
    "CPDIF-R": ("R", "XGB"),
    "CPDIF-R-LGBM": ("R", "LGBM")
}


def build_learner(name):
    learner, algorithm = MODEL_REGISTRY[name]
    if learner == "S":
        return SLearner(algorithm)
    if learner == "T":
        return TLearner(algorithm)
    if learner == "X":
        return XLearner(algorithm, PROPENSITY)
    if learner == "DR":
        return DRLearner(algorithm, PROPENSITY)
    if learner == "R":
        return RLearner(algorithm, PROPENSITY)
    raise ValueError(learner)


# ---------------------------------------------------------------------
# 6. UPLIFT METRICS AND RANDOMIZED POLICY EVALUATION
# ---------------------------------------------------------------------

def randomized_ipw_ate(T, Y):
    return float(np.mean(
        T * Y / PROPENSITY - (1 - T) * Y / (1 - PROPENSITY)
    ))




def bootstrap_ate_and_mean_ite(T, Y, ite, n_boot=N_BOOTSTRAP):
    """Treatment-stratified percentile bootstrap CIs for IPW ATE and mean ITE.

    The fitted CPDIF-S-LR model and its test-set ITE predictions are held fixed.
    Each replicate resamples control and treated test observations separately,
    with replacement and at their original stratum sizes. Equation (3) is
    recomputed using the configured randomisation propensity.
    """
    T = np.asarray(T, dtype=int).reshape(-1)
    Y = np.asarray(Y, dtype=float).reshape(-1)
    ite = np.asarray(ite, dtype=float).reshape(-1)

    if not (len(T) == len(Y) == len(ite)):
        raise ValueError("T, Y and ite must have identical lengths.")
    if len(T) == 0:
        raise ValueError("Cannot bootstrap an empty test set.")
    if not np.isin(T, [0, 1]).all():
        raise ValueError("Treatment must contain only binary values 0 and 1.")
    if not np.isfinite(Y).all() or not np.isfinite(ite).all():
        raise ValueError("Y and ite must contain only finite values.")

    control_idx = np.flatnonzero(T == 0)
    treated_idx = np.flatnonzero(T == 1)
    if len(control_idx) == 0 or len(treated_idx) == 0:
        raise ValueError("Both treatment strata must contain observations.")

    point_ipw_ate = randomized_ipw_ate(T, Y)
    point_mean_ite = float(np.mean(ite))
    rng = np.random.default_rng(SEED)
    boot_ipw_ate = np.empty(n_boot, dtype=float)
    boot_mean_ite = np.empty(n_boot, dtype=float)

    for b in range(n_boot):
        sampled_idx = np.concatenate([
            rng.choice(control_idx, size=len(control_idx), replace=True),
            rng.choice(treated_idx, size=len(treated_idx), replace=True)
        ])
        rng.shuffle(sampled_idx)
        sampled_T = T[sampled_idx]
        sampled_Y = Y[sampled_idx]
        sampled_ite = ite[sampled_idx]

        # Recompute Equation (3) in every bootstrap replicate.
        boot_ipw_ate[b] = randomized_ipw_ate(sampled_T, sampled_Y)
        # Predictions are fixed; this interval reflects test-sample resampling
        # uncertainty, not model refitting or model-selection uncertainty.
        boot_mean_ite[b] = float(np.mean(sampled_ite))

    def interval_row(metric, point, values):
        values = values[np.isfinite(values)]
        lower, upper = np.quantile(values, [0.025, 0.975])
        return {
            "Metric": metric,
            "Point_Estimate_Probability_Scale": float(point),
            "Point_Estimate_Percentage_Points": float(100 * point),
            "CI_Lower_95_Probability_Scale": float(lower),
            "CI_Upper_95_Probability_Scale": float(upper),
            "CI_Lower_95_Percentage_Points": float(100 * lower),
            "CI_Upper_95_Percentage_Points": float(100 * upper),
            "Valid_Bootstrap_Replicates": int(len(values)),
            "Requested_Bootstrap_Replicates": int(n_boot),
            "Interval_Method": "Treatment-stratified percentile bootstrap"
        }

    summary = pd.DataFrame([
        interval_row("Randomized-trial IPW ATE", point_ipw_ate, boot_ipw_ate),
        interval_row("Mean CPDIF-S-LR predicted ITE", point_mean_ite, boot_mean_ite)
    ])

    save_table(summary, "Table_I_ATE_and_Mean_ITE_Bootstrap_CI")
    pd.DataFrame({
        "Bootstrap_Replicate": np.arange(1, n_boot + 1),
        "IPW_ATE_Probability_Scale": boot_ipw_ate,
        "IPW_ATE_Percentage_Points": 100 * boot_ipw_ate,
        "Mean_Predicted_ITE_Probability_Scale": boot_mean_ite,
        "Mean_Predicted_ITE_Percentage_Points": 100 * boot_mean_ite
    }).to_csv(TABLE_DIR / "Table_I_ATE_and_Mean_ITE_Bootstrap_Replicates.csv",
              index=False)

    return summary


def uplift_metrics(Y, T, uplift):
    uplift = clean_effect(uplift)
    return {
        "Lift@30": float(uplift_at_k(
            Y, uplift, T, strategy="overall", k=0.30
        )),
        "Qini_AUC": float(qini_auc_score(Y, uplift, T)),
        "AUUC": float(uplift_auc_score(Y, uplift, T)),
        "Mean_Predicted_ITE": float(np.mean(uplift)),
        "Positive_ITE_Fraction": float(np.mean(uplift > 0))
    }


def policy_from_top_fraction(uplift, fraction=TARGET_FRACTION):
    uplift = clean_effect(uplift)
    k = max(1, int(np.ceil(len(uplift) * fraction)))
    chosen = np.argsort(uplift)[-k:]
    policy = np.zeros(len(uplift), dtype=int)
    policy[chosen] = 1
    return policy


def policy_incremental_conversions_ipw(policy, T, Y):
    """Estimated incremental conversions under a targeting policy."""
    contribution = (
        T * Y / PROPENSITY - (1 - T) * Y / (1 - PROPENSITY)
    )
    return float(np.sum(policy * contribution))


def policy_economics(policy, T, Y):
    selected = int(np.sum(policy))
    incremental_conversions = policy_incremental_conversions_ipw(
        policy, T, Y
    )
    advertising_cost = selected * AD_COST_PER_TARGETED_USER_USD
    incremental_revenue = incremental_conversions * CONVERSION_VALUE_USD
    net_incremental_value = incremental_revenue - advertising_cost
    roi = (
        net_incremental_value / advertising_cost
        if advertising_cost > 0 else np.nan
    )
    return {
        "Targeted_Users": selected,
        "Targeted_Fraction": float(np.mean(policy)),
        "Incremental_Conversions_IPW": incremental_conversions,
        "Incremental_Conversions_per_1000": (
            1000.0 * incremental_conversions / len(policy)
        ),
        "Advertising_Cost_USD": advertising_cost,
        "Incremental_Revenue_USD": incremental_revenue,
        "Net_Incremental_Value_USD": net_incremental_value,
        "ROI": roi
    }


def classify_ite(ite, positive=POSITIVE_ITE_THRESHOLD,
                 negative=NEGATIVE_ITE_THRESHOLD):
    if negative >= positive:
        raise ValueError("Negative ITE threshold must be below positive threshold.")
    ite = np.asarray(ite)
    return np.select(
        [ite > positive, ite < negative],
        ["Persuadable", "Do Not Disturb"],
        default="Intermediate"
    )


# ---------------------------------------------------------------------
# 7. ATE: REGRESSION ADJUSTMENT AND PROPENSITY-SCORE MATCHING
# ---------------------------------------------------------------------

def regression_adjusted_ate(X, T, Y):
    model = LinearRegression()
    model.fit(np.column_stack([X, T]), Y)
    return float(model.coef_[-1])


def propensity_score_matching_ate(X, T, Y, caliper=PSM_CALIPER):
    """1:1 nearest-neighbour matching on estimated propensity scores.

    Matching is performed with replacement for computational scalability.
    The caliper is on the propensity-score scale.
    """
    ps_model = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=1000, solver="lbfgs"))
    ])
    ps_model.fit(X, T)
    ps = np.clip(ps_model.predict_proba(X)[:, 1], 1e-5, 1 - 1e-5)

    treated_idx = np.where(T == 1)[0]
    control_idx = np.where(T == 0)[0]

    # Vectorized nearest-neighbour matching with replacement.
    nn = NearestNeighbors(n_neighbors=1, algorithm="auto")
    nn.fit(ps[control_idx].reshape(-1, 1))
    distances, nearest = nn.kneighbors(ps[treated_idx].reshape(-1, 1))
    distances = distances[:, 0]
    matched_control_idx = control_idx[nearest[:, 0]]
    keep = distances <= caliper
    pairs = np.column_stack([treated_idx[keep], matched_control_idx[keep]])
    distances_kept = distances[keep]

    if len(pairs) == 0:
        return np.nan, 0, np.nan

    estimate = float(np.mean(Y[pairs[:, 0]] - Y[pairs[:, 1]]))
    return estimate, len(pairs), float(np.mean(distances_kept))


# ---------------------------------------------------------------------
# 8. CAUSAL ESTIMATOR REFUTATION / ROBUSTNESS DIAGNOSTICS
# ---------------------------------------------------------------------

def run_refutation_diagnostics(data):
    """Run random-common-cause, placebo-treatment and subset diagnostics."""
    X, T, Y = data["X_test"], data["T_test"], data["Y_test"]
    rng = np.random.default_rng(SEED + 912)
    baseline = regression_adjusted_ate(X, T, Y)

    random_cause = rng.normal(size=len(Y)).reshape(-1, 1)
    common_cause = regression_adjusted_ate(
        np.column_stack([X, random_cause]), T, Y
    )
    placebo_treatment = rng.permutation(T)
    placebo = regression_adjusted_ate(X, placebo_treatment, Y)

    permutation = rng.permutation(len(Y))
    midpoint = len(Y) // 2
    subset_a = regression_adjusted_ate(
        X[permutation[:midpoint]], T[permutation[:midpoint]],
        Y[permutation[:midpoint]]
    )
    subset_b = regression_adjusted_ate(
        X[permutation[midpoint:]], T[permutation[midpoint:]],
        Y[permutation[midpoint:]]
    )

    rows = []
    for name, estimate in [
        ("Baseline regression adjustment", baseline),
        ("Random common cause added", common_cause),
        ("Placebo treatment permutation", placebo),
        ("Subset validation: half A", subset_a),
        ("Subset validation: half B", subset_b)
    ]:
        rows.append({
            "Diagnostic": name,
            "ATE": estimate,
            "ATE_Percentage_Points": 100 * estimate,
            "Difference_From_Baseline": estimate - baseline
        })

    result = pd.DataFrame(rows)
    save_table(result, "Table_Refutation_and_Robustness_Diagnostics")
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(result["Diagnostic"], result["ATE_Percentage_Points"])
    ax.axhline(0, linewidth=0.8)
    ax.set_ylabel("Estimated ATE (percentage points)")
    ax.set_title("Causal estimator refutation and robustness diagnostics")
    ax.tick_params(axis="x", rotation=20)
    save_figure(fig, "Figure_12_Refutation_and_Robustness")
    return result


# ---------------------------------------------------------------------
# 9. CONTROLLED-CONFOUNDING SELECTION MECHANISM
# ---------------------------------------------------------------------

def calibrated_logistic_intercept(linear_predictor, target_mean):
    """Numerically calibrate intercept to achieve target average retention."""
    lower, upper = -25.0, 25.0
    for _ in range(100):
        midpoint = (lower + upper) / 2
        p = 1.0 / (1.0 + np.exp(-np.clip(
            midpoint + linear_predictor, -35, 35
        )))
        if p.mean() < target_mean:
            lower = midpoint
        else:
            upper = midpoint
    return (lower + upper) / 2


def covariate_dependent_selection(X, T, strength, rng):
    """Keep all controls; probabilistically retain treated observations.

    Selection logit:
      alpha + strength * (0.8*Z(f0) + 0.5*Z(f1) - 0.4*Z(f2))
    The intercept is calibrated so expected treated retention is 70%.
    """
    standardized = []
    for column in (0, 1, 2):
        values = X[:, column]
        sd = np.std(values)
        standardized.append(
            (values - np.mean(values)) / (sd if sd > 1e-12 else 1.0)
        )

    z0, z1, z2 = standardized
    b0, b1, b2 = SELECTION_COEFFICIENTS
    linear = strength * (b0 * z0 + b1 * z1 + b2 * z2)

    treated = T == 1
    intercept = calibrated_logistic_intercept(
        linear[treated], TREATED_RETENTION_TARGET
    )
    probabilities = 1.0 / (1.0 + np.exp(-np.clip(
        intercept + linear, -35, 35
    )))

    keep = T == 0
    keep[treated] = rng.random(np.sum(treated)) < probabilities[treated]
    return keep, probabilities


def run_controlled_confounding(data):
    X, T, Y = data["X_train"], data["T_train"], data["Y_train"]
    benchmark = randomized_ipw_ate(T, Y)
    rows = []

    for strength in CONFOUNDING_STRENGTHS:
        rng = np.random.default_rng(SEED + int(strength * 1000))
        keep, selection_probability = covariate_dependent_selection(
            X, T, strength, rng
        )
        Xs, Ts, Ys = X[keep], T[keep], Y[keep]

        naive = float(Ys[Ts == 1].mean() - Ys[Ts == 0].mean())
        regression = regression_adjusted_ate(Xs, Ts, Ys)
        psm, matched, mean_distance = propensity_score_matching_ate(
            Xs, Ts, Ys
        )

        for method, estimate in (
            ("Unadjusted difference in means", naive),
            ("Regression adjustment", regression),
            ("Propensity-score matching", psm)
        ):
            absolute_bias = (
                abs(estimate - benchmark) if np.isfinite(estimate) else np.nan
            )
            rows.append({
                "Confounding_Strength": strength,
                "Selection_Coefficient_f0": SELECTION_COEFFICIENTS[0],
                "Selection_Coefficient_f1": SELECTION_COEFFICIENTS[1],
                "Selection_Coefficient_f2": SELECTION_COEFFICIENTS[2],
                "Target_Treated_Retention": TREATED_RETENTION_TARGET,
                "Observed_Treated_Retention": float(np.mean(keep[T == 1])),
                "Retained_Sample_Size": int(np.sum(keep)),
                "Method": method,
                "ATE": estimate,
                "ATE_Percentage_Points": 100 * estimate,
                "Randomized_Reference_ATE": benchmark,
                "Reference_ATE_Percentage_Points": 100 * benchmark,
                "Absolute_Bias": absolute_bias,
                "Absolute_Bias_Percentage_Points": (
                    100 * absolute_bias if np.isfinite(absolute_bias) else np.nan
                ),
                "Relative_Bias_Percent": (
                    100 * absolute_bias / abs(benchmark)
                    if np.isfinite(absolute_bias) and abs(benchmark) > 1e-12
                    else np.nan
                ),
                "PSM_Matched_Pairs": matched if method == "Propensity-score matching" else np.nan,
                "PSM_Mean_Match_Distance": (
                    mean_distance if method == "Propensity-score matching" else np.nan
                )
            })

    result = pd.DataFrame(rows)
    save_table(result, "Table_I_Controlled_Confounding")

    fig, ax = plt.subplots(figsize=(10, 6))
    for method, group in result.groupby("Method"):
        ax.plot(
            group["Confounding_Strength"],
            group["Absolute_Bias_Percentage_Points"],
            marker="o", label=method
        )
    ax.set_xlabel("Confounding strength multiplier")
    ax.set_ylabel("Absolute ATE bias (percentage points)")
    ax.set_title("Controlled-confounding experiment")
    ax.legend()
    save_figure(fig, "Figure_10_Controlled_Confounding")

    return result


# ---------------------------------------------------------------------
# 9. BOOTSTRAP INTERVALS FOR POLICY VALUE AND ROI
# ---------------------------------------------------------------------

def bootstrap_policy_economics(ite, T, Y, n_boot=N_BOOTSTRAP):
    rng = np.random.default_rng(SEED)
    control_idx = np.where(T == 0)[0]
    treated_idx = np.where(T == 1)[0]

    def calculate(indices):
        sampled_ite = ite[indices]
        sampled_t = T[indices]
        sampled_y = Y[indices]
        segments = classify_ite(sampled_ite)
        policy = (segments == "Persuadable").astype(int)
        return policy_economics(policy, sampled_t, sampled_y)

    point_segments = classify_ite(ite)
    point_policy = (point_segments == "Persuadable").astype(int)
    point = policy_economics(point_policy, T, Y)

    boot_rows = []
    for b in range(n_boot):
        sample = np.concatenate([
            rng.choice(control_idx, size=len(control_idx), replace=True),
            rng.choice(treated_idx, size=len(treated_idx), replace=True)
        ])
        rng.shuffle(sample)
        boot_rows.append(calculate(sample))

    boot = pd.DataFrame(boot_rows)
    metrics = [
        "Targeted_Fraction",
        "Incremental_Conversions_per_1000",
        "Advertising_Cost_USD",
        "Incremental_Revenue_USD",
        "Net_Incremental_Value_USD",
        "ROI"
    ]

    rows = []
    for metric in metrics:
        values = pd.to_numeric(boot[metric], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        ).dropna()
        rows.append({
            "Metric": metric,
            "Point_Estimate": point[metric],
            "CI_Lower_95": values.quantile(0.025),
            "CI_Upper_95": values.quantile(0.975),
            "Valid_Bootstrap_Replicates": int(len(values)),
            "Requested_Bootstrap_Replicates": n_boot
        })

    table = pd.DataFrame(rows)
    save_table(table, "Table_II_Bootstrap_Policy_Economics")
    boot.to_csv(TABLE_DIR / "Table_II_Bootstrap_Replicate_Values.csv", index=False)
    return point, table


# ---------------------------------------------------------------------
# 10. THRESHOLD SENSITIVITY
# ---------------------------------------------------------------------

def run_threshold_sensitivity(ite, T, Y):
    rows = []
    for magnitude in THRESHOLD_MAGNITUDES:
        positive = magnitude
        negative = -magnitude
        segment = classify_ite(ite, positive, negative)
        policy = (segment == "Persuadable").astype(int)
        economics = policy_economics(policy, T, Y)

        rows.append({
            "Positive_ITE_Threshold": positive,
            "Negative_ITE_Threshold": negative,
            "Persuadable_Users": int(np.sum(segment == "Persuadable")),
            "Persuadable_Users_Percent": 100 * np.mean(segment == "Persuadable"),
            "Intermediate_Users_Percent": 100 * np.mean(segment == "Intermediate"),
            "Do_Not_Disturb_Users_Percent": 100 * np.mean(segment == "Do Not Disturb"),
            "Advertising_Exposure_Reduction_Percent": 100 * (1 - np.mean(policy)),
            "Incremental_Conversions_IPW": economics["Incremental_Conversions_IPW"],
            "Incremental_Conversions_Per_1000": economics["Incremental_Conversions_per_1000"],
            "Advertising_Cost_USD": economics["Advertising_Cost_USD"],
            "Incremental_Revenue_USD": economics["Incremental_Revenue_USD"],
            "Net_Incremental_Value_USD": economics["Net_Incremental_Value_USD"],
            "ROI": economics["ROI"]
        })

    result = pd.DataFrame(rows)
    save_table(result, "Table_III_ITE_Threshold_Sensitivity")

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(
        result["Positive_ITE_Threshold"],
        result["Persuadable_Users_Percent"],
        marker="o", label="Persuadable users (%)"
    )
    ax1.set_xlabel("Absolute ITE threshold")
    ax1.set_ylabel("Persuadable users (%)")
    ax1.set_title("Sensitivity to positive and negative ITE thresholds")
    ax2 = ax1.twinx()
    ax2.plot(
        result["Positive_ITE_Threshold"],
        result["ROI"], marker="s", linestyle="--", label="ROI"
    )
    ax2.set_ylabel("ROI")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
    save_figure(fig, "Figure_11_ITE_Threshold_Sensitivity")
    return result


# ---------------------------------------------------------------------
# 11. PAIRED BOOTSTRAP MODEL COMPARISONS
# ---------------------------------------------------------------------

def paired_bootstrap_metric_difference(Y, T, prediction_a, prediction_b,
                                       metric_name, n_boot=N_BOOTSTRAP):
    rng = np.random.default_rng(SEED)
    idx0 = np.where(T == 0)[0]
    idx1 = np.where(T == 1)[0]

    def metric(y, t, pred):
        if metric_name == "Qini_AUC":
            return qini_auc_score(y, pred, t)
        if metric_name == "AUUC":
            return uplift_auc_score(y, pred, t)
        if metric_name == "Lift@30":
            return uplift_at_k(y, pred, t, strategy="overall", k=0.30)
        raise ValueError(metric_name)

    differences = []
    for _ in range(n_boot):
        sample = np.concatenate([
            rng.choice(idx0, len(idx0), replace=True),
            rng.choice(idx1, len(idx1), replace=True)
        ])
        rng.shuffle(sample)
        try:
            a = metric(Y[sample], T[sample], prediction_a[sample])
            b = metric(Y[sample], T[sample], prediction_b[sample])
            difference = float(a - b)
            if np.isfinite(difference):
                differences.append(difference)
        except Exception:
            continue

    values = np.asarray(differences)
    if len(values) < max(100, int(0.8 * n_boot)):
        warnings.warn(
            f"Only {len(values)} valid bootstrap replicates for {metric_name}."
        )

    return {
        "Metric": metric_name,
        "Mean_Difference_A_minus_B": float(np.mean(values)) if len(values) else np.nan,
        "CI_Lower_95": float(np.quantile(values, 0.025)) if len(values) else np.nan,
        "CI_Upper_95": float(np.quantile(values, 0.975)) if len(values) else np.nan,
        "Valid_Replicates": int(len(values))
    }


# ---------------------------------------------------------------------
# 12. FIGURES: ARCHITECTURE, DESCRIPTIVES, SCM, SHAP
# ---------------------------------------------------------------------

def generate_architecture_figure():
    fig, ax = plt.subplots(figsize=(10, 11))
    ax.axis("off")
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)

    stages = [
        ("Data Layer", "Criteo covariates X, treatment T and conversion Y"),
        ("Data Preparation", "Sampling, data validation, imputation and partitioning"),
        ("Causal Identification", "SCM: X → T, X → Y and T → Y; backdoor adjustment"),
        ("Causal Effect Estimation", "Regression adjustment and propensity-score matching\nCounterfactual outcome model f(X,T)"),
        ("Robustness and Validation", "Controlled-confounding, placebo/refutation and subset checks"),
        ("Customer Response Modelling", "CPDIF-S-LR counterfactual predictions Y(1), Y(0) and ITE = Y(1) − Y(0)"),
        ("Prescriptive Decision Engine", "Persuadable / Intermediate / Do Not Disturb\nPositive-ITE targeting, exposure and ROI evaluation"),
        ("Decision Outputs", "ATE, ITE, customer segments, target policy and economic outcomes")
    ]

    ys = np.linspace(10.9, 1.0, len(stages))
    for (title, description), y in zip(stages, ys):
        ax.text(
            5, y, f"{title}\n{description}",
            ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.6", fc="#eaf1f8", ec="#34495e")
        )
    for y1, y2 in zip(ys[:-1], ys[1:]):
        ax.annotate(
            "", xy=(5, y2 + 0.43), xytext=(5, y1 - 0.43),
            arrowprops=dict(arrowstyle="->", lw=1.2)
        )

    ax.text(
        8.3, 8.7, "Primary outcome model\nCPDIF-S-LR + SHAP\nPredictive explanation only",
        ha="center", va="center", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray")
    )
    ax.set_title("Causal-Prescriptive Decision Intelligence Framework (CPDIF)",
                 fontsize=13, weight="bold")
    save_figure(fig, "Figure_1_CPDIF_Architecture")


def generate_descriptive_outputs(frame):
    T = frame[TREATMENT_COL].to_numpy()
    Y = frame[OUTCOME_COL].to_numpy()

    rows = [
        ("Total observations", len(frame)),
        ("Control observations", int(np.sum(T == 0))),
        ("Treated observations", int(np.sum(T == 1))),
        ("Control proportion (%)", 100 * np.mean(T == 0)),
        ("Treated proportion (%)", 100 * np.mean(T == 1)),
        ("Total conversions", int(np.sum(Y == 1))),
        ("Overall conversion rate (%)", 100 * np.mean(Y)),
        ("Control conversion rate (%)", 100 * np.mean(Y[T == 0])),
        ("Treated conversion rate (%)", 100 * np.mean(Y[T == 1])),
        ("Naive difference (percentage points)",
         100 * (np.mean(Y[T == 1]) - np.mean(Y[T == 0])))
    ]
    save_table(
        pd.DataFrame(rows, columns=["Measure", "Value"]),
        "Descriptive_Statistics"
    )

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    panels = [
        (["Control", "Treated"],
         [100 * np.mean(T == 0), 100 * np.mean(T == 1)],
         "Treatment allocation", "Users (%)"),
        (["No conversion", "Conversion"],
         [100 * np.mean(Y == 0), 100 * np.mean(Y == 1)],
         "Conversion distribution", "Users (%)"),
        (["Control", "Treated"],
         [100 * np.mean(Y[T == 0]), 100 * np.mean(Y[T == 1])],
         "Conversion rate by treatment", "Conversion rate (%)")
    ]

    for ax, (labels, values, title, ylabel) in zip(axes, panels):
        bars = ax.bar(labels, values)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        for bar, value in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(), f"{value:.4f}",
                ha="center", va="bottom", fontsize=8
            )
    save_figure(fig, "Figure_2_Descriptive_Characteristics")


def generate_scm_figure():
    graph = nx.DiGraph()
    graph.add_nodes_from(FEATURES + ["Treatment", "Conversion"])
    for feature in FEATURES:
        graph.add_edge(feature, "Treatment")
        graph.add_edge(feature, "Conversion")
    graph.add_edge("Treatment", "Conversion")

    fig, ax = plt.subplots(figsize=(12, 8))
    positions = nx.spring_layout(graph, seed=SEED, k=0.9)
    nx.draw_networkx(
        graph, positions, ax=ax, node_size=1400,
        font_size=8, arrows=True, arrowsize=14
    )
    ax.set_title("Structural Causal Model for Treatment and Conversion")
    ax.axis("off")
    save_figure(fig, "Figure_3_Structural_Causal_Model")


def generate_shap_outputs(model, data):
    """Explain the primary logistic-regression outcome model associationally."""
    try:
        import shap
    except ImportError:
        print("SHAP is not installed. Skipping SHAP figures and table.")
        return

    rng = np.random.default_rng(SEED)
    n = min(SHAP_SAMPLE_SIZE, len(data["X_test"]))
    selected = rng.choice(len(data["X_test"]), size=n, replace=False)
    X_eval_raw = np.column_stack([
        data["X_test"][selected], data["T_test"][selected]
    ])
    feature_names = FEATURES + ["treatment"]

    # CPDIF-S-LR is a sklearn Pipeline(StandardScaler, LogisticRegression).
    if isinstance(model, Pipeline):
        scaler = model.named_steps["scaler"]
        classifier = model.named_steps["classifier"]
        X_eval = scaler.transform(X_eval_raw)
        background_raw = np.column_stack([
            data["X_train"][:min(1000, len(data["X_train"]))],
            data["T_train"][:min(1000, len(data["T_train"]))]
        ])
        background = scaler.transform(background_raw)
        explainer = shap.LinearExplainer(classifier, background)
        shap_values = explainer.shap_values(X_eval)
        prediction_for_local = predict_probability_one(model, X_eval_raw)
        explanation_space = X_eval
        explanation_note = "Logistic-regression log-odds contribution; predictive association, not causal attribution"
    else:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_eval_raw)
        prediction_for_local = predict_probability_one(model, X_eval_raw)
        explanation_space = X_eval_raw
        explanation_note = "Predictive model contribution; not a causal attribution"

    if isinstance(shap_values, list):
        shap_values = shap_values[-1]
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    importance = pd.DataFrame({
        "Feature": feature_names,
        "Mean_Absolute_SHAP": np.mean(np.abs(shap_values), axis=0),
        "Interpretation": explanation_note
    }).sort_values("Mean_Absolute_SHAP", ascending=False)
    save_table(importance, "SHAP_Feature_Importance")

    fig, ax = plt.subplots(figsize=(9, 6))
    ordered = importance.sort_values("Mean_Absolute_SHAP")
    ax.barh(ordered["Feature"], ordered["Mean_Absolute_SHAP"])
    ax.set_xlabel("Mean absolute SHAP value")
    ax.set_title("CPDIF-S-LR predictive feature contributions (SHAP)")
    save_figure(fig, "Figure_4_SHAP_Feature_Importance")

    plt.figure(figsize=(9, 7))
    shap.summary_plot(
        shap_values, explanation_space, feature_names=feature_names,
        show=False, max_display=len(feature_names)
    )
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "Figure_5_SHAP_Summary.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / "Figure_5_SHAP_Summary.pdf", bbox_inches="tight")
    plt.close()

    local_index = int(np.argsort(prediction_for_local)[len(prediction_for_local) // 2])
    try:
        if isinstance(model, Pipeline):
            local_exp = explainer(X_eval[local_index:local_index + 1])
        else:
            local_exp = explainer(X_eval_raw[local_index:local_index + 1])
        shap.plots.waterfall(local_exp[0], max_display=12, show=False)
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / "Figure_6_SHAP_Local_Explanation.png", dpi=300, bbox_inches="tight")
        plt.savefig(FIGURE_DIR / "Figure_6_SHAP_Local_Explanation.pdf", bbox_inches="tight")
        plt.close()
    except Exception as error:
        print(f"Local SHAP waterfall could not be generated: {error}")

    print("SHAP outputs generated. They explain predictive associations, not causal effects.")


# ---------------------------------------------------------------------
# 13. COUNTERFACTUAL ITE FIGURES AND CUSTOMER SEGMENTS
# ---------------------------------------------------------------------

def generate_ite_outputs(ite, p0, p1):
    segments = classify_ite(ite)
    order = ["Persuadable", "Intermediate", "Do Not Disturb"]
    counts = pd.Series(segments).value_counts().reindex(order, fill_value=0)

    segment_table = pd.DataFrame({
        "Segment": order,
        "Users": counts.values,
        "Percentage": 100 * counts.values / len(ite)
    })
    save_table(segment_table, "ITE_Customer_Segments")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].hist(ite, bins=100)
    axes[0].axvline(
        POSITIVE_ITE_THRESHOLD, linestyle="--",
        label="Persuadable threshold"
    )
    axes[0].axvline(
        NEGATIVE_ITE_THRESHOLD, linestyle="--",
        label="Do Not Disturb threshold"
    )
    axes[0].axvline(0, color="black", linewidth=0.8)
    axes[0].set_xlabel("Estimated ITE (conversion probability difference)")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title("Distribution of Individual Treatment Effects")
    axes[0].legend()

    axes[1].bar(order, counts.values)
    axes[1].set_ylabel("Number of users")
    axes[1].set_title("Customer response segments")
    axes[1].tick_params(axis="x", rotation=15)
    save_figure(fig, "Figure_7_ITE_Distribution_and_Segments")

    # Five observations closest to selected ITE quantiles.
    quantiles = (0.99, 0.75, 0.50, 0.25, 0.01)
    chosen = [
        int(np.argmin(np.abs(ite - np.quantile(ite, q))))
        for q in quantiles
    ]
    customer_rows = []
    for label, quantile, index in zip("ABCDE", quantiles, chosen):
        customer_rows.append({
            "Customer_Profile": label,
            "ITE_Quantile": quantile,
            "Predicted_Conversion_Without_Treatment": p0[index],
            "Predicted_Conversion_With_Treatment": p1[index],
            "Predicted_ITE": ite[index],
            "Segment": segments[index]
        })

    customer_table = pd.DataFrame(customer_rows)
    save_table(customer_table, "Five_Counterfactual_Customer_Profiles")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    x = np.arange(len(customer_table))
    axes[0].bar(
        x - 0.18,
        customer_table["Predicted_Conversion_Without_Treatment"],
        width=0.36, label="Without treatment"
    )
    axes[0].bar(
        x + 0.18,
        customer_table["Predicted_Conversion_With_Treatment"],
        width=0.36, label="With treatment"
    )
    axes[0].set_xticks(x, customer_table["Customer_Profile"])
    axes[0].set_ylabel("Predicted conversion probability")
    axes[0].set_title("Counterfactual conversion predictions")
    axes[0].legend()

    axes[1].bar(
        customer_table["Customer_Profile"],
        customer_table["Predicted_ITE"]
    )
    axes[1].axhline(POSITIVE_ITE_THRESHOLD, linestyle="--")
    axes[1].axhline(NEGATIVE_ITE_THRESHOLD, linestyle="--")
    axes[1].set_ylabel("Predicted ITE")
    axes[1].set_title("Estimated treatment effects for five profiles")
    save_figure(fig, "Figure_8_Five_Counterfactual_Profiles")


# ---------------------------------------------------------------------
# 14. MODEL BENCHMARK
# ---------------------------------------------------------------------

def run_model_benchmark(data):
    validation_rows = []
    test_rows = []
    validation_predictions = {}
    test_predictions = {}
    runtimes = {}

    for model_name in MODEL_REGISTRY:
        print(f"\nFitting {model_name}")
        start = time.time()

        learner = build_learner(model_name)
        learner.fit(
            data["X_train"], data["T_train"], data["Y_train"]
        )
        elapsed = time.time() - start
        runtimes[model_name] = elapsed

        val_prediction = clean_effect(
            learner.predict_uplift(data["X_validation"])
        )
        test_prediction = clean_effect(
            learner.predict_uplift(data["X_test"])
        )

        val_metrics = uplift_metrics(
            data["Y_validation"], data["T_validation"], val_prediction
        )
        test_metrics = uplift_metrics(
            data["Y_test"], data["T_test"], test_prediction
        )

        val_metrics.update({"Model": model_name, "Fit_Time_Seconds": elapsed})
        test_metrics.update({"Model": model_name, "Fit_Time_Seconds": elapsed})

        validation_rows.append(val_metrics)
        test_rows.append(test_metrics)
        validation_predictions[model_name] = val_prediction
        test_predictions[model_name] = test_prediction

        joblib.dump(learner, MODEL_DIR / f"{model_name}.joblib")
        print(
            f"  Validation Qini={val_metrics['Qini_AUC']:.6f}; "
            f"Test Qini={test_metrics['Qini_AUC']:.6f}; "
            f"fit={elapsed:.1f}s"
        )
        del learner
        gc.collect()

    validation = pd.DataFrame(validation_rows).sort_values(
        "Qini_AUC", ascending=False
    ).reset_index(drop=True)
    test = pd.DataFrame(test_rows).sort_values(
        "Qini_AUC", ascending=False
    ).reset_index(drop=True)

    validation.to_csv(OUTPUT_DIR / "CPDIF_Validation_Results.csv", index=False)
    test.to_csv(OUTPUT_DIR / "CPDIF_Test_Results.csv", index=False)
    pd.DataFrame(validation_predictions).to_csv(
        OUTPUT_DIR / "Validation_Uplift_Predictions.csv", index=False
    )
    pd.DataFrame(test_predictions).to_csv(
        OUTPUT_DIR / "Test_Uplift_Predictions.csv", index=False
    )

    # A combined validation/test comparison table.
    validation_cols = ["Model", "Lift@30", "Qini_AUC", "AUUC"]
    val_table = validation[validation_cols].rename(
        columns={c: f"Validation_{c}" for c in validation_cols if c != "Model"}
    )
    test_table = test[validation_cols].rename(
        columns={c: f"Test_{c}" for c in validation_cols if c != "Model"}
    )
    comparison = val_table.merge(test_table, on="Model")
    comparison = comparison.sort_values(
        "Validation_Qini_AUC", ascending=False
    )
    save_table(comparison, "Table_IV_12_Model_Comparison")

    # Metric comparison charts. Test metrics are descriptive final evaluation.
    for metric, title in (
        ("Qini_AUC", "Qini AUC"),
        ("AUUC", "AUUC"),
        ("Lift@30", "Lift@30")
    ):
        ordered = test.sort_values(metric, ascending=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(ordered["Model"], ordered[metric])
        ax.set_xlabel(title)
        ax.set_title(f"Test-set uplift model comparison: {title}")
        save_figure(fig, f"Model_Comparison_{metric.replace('@', '_')}")

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.boxplot(
        [test_predictions[name] for name in MODEL_REGISTRY],
        labels=list(MODEL_REGISTRY), showfliers=False
    )
    ax.set_ylabel("Predicted individual treatment effect")
    ax.set_title("Test-set ITE distributions across estimator configurations")
    ax.tick_params(axis="x", rotation=40)
    save_figure(fig, "Model_Comparison_ITE_Distributions")

    # Paired bootstrap: validation-selected estimator versus CPDIF-S.
    selected_model = str(validation.iloc[0]["Model"])
    bootstrap_rows = []
    if selected_model != "CPDIF-S":
        for metric in ("Qini_AUC", "AUUC", "Lift@30"):
            result = paired_bootstrap_metric_difference(
                data["Y_test"], data["T_test"],
                test_predictions[selected_model],
                test_predictions["CPDIF-S"],
                metric
            )
            result["Model_A"] = selected_model
            result["Model_B"] = "CPDIF-S"
            bootstrap_rows.append(result)

    if bootstrap_rows:
        save_table(
            pd.DataFrame(bootstrap_rows),
            "Paired_Bootstrap_Selected_Model_vs_CPDIF_S"
        )

    return validation, test, validation_predictions, test_predictions


# ---------------------------------------------------------------------
# 15. ASSOCIATION VERSUS CAUSAL ESTIMATES
# ---------------------------------------------------------------------

def generate_association_causal_comparison(data, cp_model, cp_ite):
    Xtr_t = np.column_stack([data["X_train"], data["T_train"]])
    Xte_t = np.column_stack([data["X_test"], data["T_test"]])

    # Standardized logistic-regression coefficients for the primary CPDIF-S-LR model.
    if isinstance(cp_model.model, Pipeline):
        classifier = cp_model.model.named_steps["classifier"]
        coefficients = np.asarray(classifier.coef_).reshape(-1)
        coefficient_table = pd.DataFrame({
            "Feature": FEATURES + ["treatment"],
            "Standardized_Logistic_Coefficient": coefficients,
            "Absolute_Coefficient": np.abs(coefficients)
        }).sort_values("Absolute_Coefficient", ascending=False)
        save_table(coefficient_table, "CPDIF_S_LR_Standardized_Coefficients")

    regression_ate = regression_adjusted_ate(
        data["X_test"], data["T_test"], data["Y_test"]
    )
    psm_ate, matched_pairs, mean_distance = propensity_score_matching_ate(
        data["X_test"], data["T_test"], data["Y_test"]
    )
    randomized_ate = randomized_ipw_ate(
        data["T_test"], data["Y_test"]
    )

    # 1,000 treatment-stratified bootstrap replicates for the IPW ATE and
    # the mean fixed CPDIF-S-LR predicted ITE on the same held-out test set.
    bootstrap_ci = bootstrap_ate_and_mean_ite(
        data["T_test"], data["Y_test"], cp_ite, n_boot=N_BOOTSTRAP
    )
    ipw_ci = bootstrap_ci.loc[
        bootstrap_ci["Metric"] == "Randomized-trial IPW ATE"
    ].iloc[0]
    ite_ci = bootstrap_ci.loc[
        bootstrap_ci["Metric"] == "Mean CPDIF-S-LR predicted ITE"
    ].iloc[0]

    estimates = pd.DataFrame([
        {
            "Estimator": "Randomized-trial IPW ATE",
            "ATE_Probability_Scale": randomized_ate,
            "ATE_Percentage_Points": 100 * randomized_ate,
            "Matched_Pairs": np.nan
        },
        {
            "Estimator": "Regression adjustment",
            "ATE_Probability_Scale": regression_ate,
            "ATE_Percentage_Points": 100 * regression_ate,
            "Matched_Pairs": np.nan
        },
        {
            "Estimator": "Propensity-score matching",
            "ATE_Probability_Scale": psm_ate,
            "ATE_Percentage_Points": 100 * psm_ate,
            "Matched_Pairs": matched_pairs
        },
        {
            "Estimator": "Mean CPDIF-S-LR counterfactual ITE",
            "ATE_Probability_Scale": float(np.mean(cp_ite)),
            "ATE_Percentage_Points": 100 * float(np.mean(cp_ite)),
            "Matched_Pairs": np.nan
        }
    ])
    estimates["CI_Lower_95_Percentage_Points"] = np.nan
    estimates["CI_Upper_95_Percentage_Points"] = np.nan
    estimates.loc[
        estimates["Estimator"] == "Randomized-trial IPW ATE",
        "CI_Lower_95_Percentage_Points"
    ] = ipw_ci["CI_Lower_95_Percentage_Points"]
    estimates.loc[
        estimates["Estimator"] == "Randomized-trial IPW ATE",
        "CI_Upper_95_Percentage_Points"
    ] = ipw_ci["CI_Upper_95_Percentage_Points"]
    estimates.loc[
        estimates["Estimator"] == "Mean CPDIF-S-LR counterfactual ITE",
        "CI_Lower_95_Percentage_Points"
    ] = ite_ci["CI_Lower_95_Percentage_Points"]
    estimates.loc[
        estimates["Estimator"] == "Mean CPDIF-S-LR counterfactual ITE",
        "CI_Upper_95_Percentage_Points"
    ] = ite_ci["CI_Upper_95_Percentage_Points"]

    estimates["PSM_Mean_Match_Distance"] = np.nan
    estimates.loc[
        estimates["Estimator"] == "Propensity-score matching",
        "PSM_Mean_Match_Distance"
    ] = mean_distance
    save_table(estimates, "Causal_Effect_Estimates")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(estimates["Estimator"], estimates["ATE_Percentage_Points"])
    ax.set_ylabel("ATE (percentage points)")
    ax.set_title("Comparison of causal effect estimates")
    ax.tick_params(axis="x", rotation=20)
    save_figure(fig, "Figure_9_Association_and_Causal_Estimates")


# ---------------------------------------------------------------------
# 16. MAIN EXPERIMENT PIPELINE
# ---------------------------------------------------------------------

def main():
    if not (0 < PROPENSITY < 1):
        raise ValueError("PROPENSITY must be strictly between 0 and 1.")
    if not np.isclose(
        TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION, 1.0
    ):
        raise ValueError("Train, validation and test fractions must sum to 1.")
    if CONVERSION_VALUE_USD <= 0 or AD_COST_PER_TARGETED_USER_USD <= 0:
        raise ValueError("Economic values must be positive.")

    print("=" * 78)
    print("CPDIF-S-LR COMPLETE END-TO-END EXPERIMENT")
    print("=" * 78)

    # A. Data and split
    frame = load_criteo_data()
    data = create_data_splits(frame)

    print("\nData diagnostics")
    print("----------------")
    print(f"Analytical observations: {len(frame):,}")
    print(
        "Train/validation/test:",
        len(data["Y_train"]), len(data["Y_validation"]), len(data["Y_test"])
    )
    print(f"Observed treatment fraction: {frame[TREATMENT_COL].mean():.6f}")
    print(f"Configured assignment propensity: {PROPENSITY:.6f}")
    print(f"Overall conversion rate: {frame[OUTCOME_COL].mean():.6%}")
    print(
        "Control conversion rate:",
        f"{frame.loc[frame[TREATMENT_COL] == 0, OUTCOME_COL].mean():.6%}"
    )
    print(
        "Treated conversion rate:",
        f"{frame.loc[frame[TREATMENT_COL] == 1, OUTCOME_COL].mean():.6%}"
    )

    # B. Benchmark the candidate estimators; selection is based only on validation Qini.
    print("\nRunning the 12-configuration estimator benchmark")
    validation_results, test_results, val_predictions, test_predictions = (
        run_model_benchmark(data)
    )
    validation_winner = str(validation_results.iloc[0]["Model"])
    print(f"Highest validation Qini configuration: {validation_winner}")

    # C. CPDIF-S-LR is the prespecified primary model for all downstream analyses.
    # Fit on train + validation only; the test set remains reserved for evaluation.
    PRIMARY_MODEL_NAME = "CPDIF-S-LR"
    if PRIMARY_MODEL_NAME not in MODEL_REGISTRY:
        raise RuntimeError("CPDIF-S-LR is not registered in MODEL_REGISTRY.")
    print("\nFitting the single primary CPDIF-S-LR model (logistic-regression S-learner)")
    cp_model = SLearner("LR")
    X_fit = np.vstack([data["X_train"], data["X_validation"]])
    T_fit = np.concatenate([data["T_train"], data["T_validation"]])
    Y_fit = np.concatenate([data["Y_train"], data["Y_validation"]])
    cp_model.fit(X_fit, T_fit, Y_fit)
    joblib.dump(cp_model, MODEL_DIR / "CPDIF_S_LR_Primary_Model.joblib")

    p0, p1, cp_ite = cp_model.predict_counterfactuals(data["X_test"])
    cp_ite = clean_effect(cp_ite)
    np.savez_compressed(
        OUTPUT_DIR / "CPDIF_S_LR_Test_Counterfactual_Predictions.npz",
        p0=p0, p1=p1, ite=cp_ite, test_idx=data["test_idx"]
    )
    pd.DataFrame({
        "Test_Row_Index": data["test_idx"],
        "Predicted_Conversion_T0": p0,
        "Predicted_Conversion_T1": p1,
        "Predicted_ITE": cp_ite,
        "ITE_Segment": classify_ite(cp_ite)
    }).to_csv(OUTPUT_DIR / "CPDIF_S_LR_Test_ITE_Predictions.csv", index=False)

    # D. Main framework figures, descriptive analysis and SHAP.
    generate_architecture_figure()
    generate_descriptive_outputs(frame)
    generate_scm_figure()
    generate_shap_outputs(cp_model.model, data)

    # E. ATE comparisons, ITE segmentation and representative profiles.
    generate_association_causal_comparison(data, cp_model, cp_ite)
    generate_ite_outputs(cp_ite, p0, p1)

    # F. Policy evaluation and ROI.
    cp_segment = classify_ite(cp_ite)
    cp_policy = (cp_segment == "Persuadable").astype(int)
    top_fraction_policy = policy_from_top_fraction(cp_ite, TARGET_FRACTION)
    broad_policy = np.ones(len(cp_ite), dtype=int)

    policy_rows = []
    for policy_name, policy in (
        ("CPDIF-S-LR positive-ITE threshold policy", cp_policy),
        ("CPDIF-S-LR top-8.7-percent policy", top_fraction_policy),
        ("Broad targeting policy", broad_policy),
        ("No-treatment policy", np.zeros(len(cp_ite), dtype=int))
    ):
        row = {"Policy": policy_name}
        row.update(policy_economics(policy, data["T_test"], data["Y_test"]))
        policy_rows.append(row)

    policy_table = pd.DataFrame(policy_rows)
    save_table(policy_table, "Policy_Evaluation_and_ROI")

    point_economics, bootstrap_table = bootstrap_policy_economics(
        cp_ite, data["T_test"], data["Y_test"]
    )
    threshold_table = run_threshold_sensitivity(
        cp_ite, data["T_test"], data["Y_test"]
    )

    # G. Refutation and controlled-confounding analyses.
    refutation_table = run_refutation_diagnostics(data)
    confounding_table = run_controlled_confounding(data)

    # H. Save experiment configuration and summary.
    metadata = {
        "dataset": DATASET_ID,
        "seed": SEED,
        "sample_fraction": SAMPLE_FRACTION,
        "split": {
            "train": TRAIN_FRACTION,
            "validation": VALIDATION_FRACTION,
            "test": TEST_FRACTION
        },
        "features": FEATURES,
        "excluded_columns": ["visit", "exposure"],
        "assignment_propensity": PROPENSITY,
        "primary_cp_model": "CPDIF-S-LR: S-learner with standardized logistic-regression outcome model",
        "primary_model_fit_data": "training plus validation partitions; test partition reserved for evaluation",
        "validation_selected_benchmark_model": validation_winner,
        "benchmark_configurations": MODEL_REGISTRY,
        "selection_metric_for_benchmark": "Validation Qini_AUC",
        "controlled_confounding": {
            "selection_coefficients": SELECTION_COEFFICIENTS,
            "treated_retention_target": TREATED_RETENTION_TARGET,
            "strengths": CONFOUNDING_STRENGTHS
        },
        "economic_configuration": {
            "conversion_value_usd": CONVERSION_VALUE_USD,
            "advertising_cost_per_targeted_user_usd": AD_COST_PER_TARGETED_USER_USD
        },
        "ite_thresholds": {
            "positive": POSITIVE_ITE_THRESHOLD,
            "negative": NEGATIVE_ITE_THRESHOLD,
            "sensitivity_magnitudes": THRESHOLD_MAGNITUDES
        },
        "bootstrap_replicates": N_BOOTSTRAP,
        "psm_caliper": PSM_CALIPER
    }

    with open(OUTPUT_DIR / "Experiment_Configuration.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    summary = {
        "sample_size": int(len(frame)),
        "train_size": int(len(data["Y_train"])),
        "validation_size": int(len(data["Y_validation"])),
        "test_size": int(len(data["Y_test"])),
        "observed_treatment_fraction": float(frame[TREATMENT_COL].mean()),
        "overall_conversion_rate": float(frame[OUTCOME_COL].mean()),
        "randomized_ipw_ate_test": randomized_ipw_ate(
            data["T_test"], data["Y_test"]
        ),
        "primary_model": "CPDIF-S-LR",
        "primary_model_validation_rank": int(np.flatnonzero(validation_results["Model"].to_numpy() == "CPDIF-S-LR")[0] + 1),
        "cpdif_s_lr_mean_ite": float(np.mean(cp_ite)),
        "cpdif_s_lr_mean_ite_percentage_points": 100 * float(np.mean(cp_ite)),
        "cpdif_s_lr_ite_sd": float(np.std(cp_ite, ddof=1)),
        "cpdif_s_lr_persuadable_fraction": float(np.mean(cp_segment == "Persuadable")),
        "cpdif_s_lr_intermediate_fraction": float(np.mean(cp_segment == "Intermediate")),
        "cpdif_s_lr_do_not_disturb_fraction": float(np.mean(cp_segment == "Do Not Disturb")),
        "cpdif_s_lr_threshold_policy_economics": point_economics,
        "best_validation_benchmark_model": str(validation_results.iloc[0]["Model"]),
        "best_validation_qini_auc": float(validation_results.iloc[0]["Qini_AUC"])
    }

    with open(OUTPUT_DIR / "Run_Summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\n" + "=" * 78)
    print("CPDIF-S-LR EXPERIMENT PIPELINE FINISHED")
    print("=" * 78)
    print(f"Outputs: {OUTPUT_DIR.resolve()}")
    print(f"Figures: {FIGURE_DIR.resolve()}")
    print(f"Tables: {TABLE_DIR.resolve()}")
    print(f"Models: {MODEL_DIR.resolve()}")
    print("\nCPDIF-S-LR test-set ITE mean (percentage points):",
          summary["cpdif_s_lr_mean_ite_percentage_points"])
    print("CPDIF-S-LR persuadable fraction:",
          summary["cpdif_s_lr_persuadable_fraction"])
    print("Primary downstream model: CPDIF-S-LR")
    print("Best validation benchmark configuration:",
          summary["best_validation_benchmark_model"])
    print("\nReview Experiment_Configuration.json and Run_Summary.json.")


if __name__ == "__main__":
    main()


CPDIF-S-LR COMPLETE END-TO-END EXPERIMENT
Loading dataset: criteo/criteo-uplift


README.md:   0%|          | 0.00/2.68k [00:00<?, ?B/s]

criteo-research-uplift-v2.1.csv.gz: reconstructing file:   0%|          |  0.00B /  311MB            

criteo-research-uplift-v2.1.csv.gz: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/13979592 [00:00<?, ? examples/s]

Dataset rows: 13,979,592; sample rows: 1,397,959

Data diagnostics
----------------
Analytical observations: 1,397,959
Train/validation/test: 978571 209694 209694
Observed treatment fraction: 0.849698
Configured assignment propensity: 0.850000
Overall conversion rate: 0.295502%
Control conversion rate: 0.190371%
Treated conversion rate: 0.314099%

Running the 12-configuration estimator benchmark

Fitting CPDIF-S
  Validation Qini=0.185742; Test Qini=0.231271; fit=33.6s

Fitting CPDIF-S-LGBM
  Validation Qini=0.185190; Test Qini=0.206400; fit=38.9s

Fitting CPDIF-S-ET
  Validation Qini=0.035070; Test Qini=0.114600; fit=599.4s

Fitting CPDIF-S-LR
  Validation Qini=0.193032; Test Qini=0.210463; fit=3.9s

Fitting CPDIF-TXGB
  Validation Qini=0.003625; Test Qini=0.095433; fit=31.3s

Fitting CPDIF-TLGBM
  Validation Qini=0.108207; Test Qini=0.153486; fit=37.4s

Fitting CPDIF-X
  Validation Qini=0.190420; Test Qini=0.145099; fit=59.8s

Fitting CPDIF-X-LGBM
  Validation Qini=0.101586; Test Qin

SHAP outputs generated. They explain predictive associations, not causal effects.

CPDIF-S-LR EXPERIMENT PIPELINE FINISHED
Outputs: /content/CPDIF_Results
Figures: /content/CPDIF_Results/Figures
Tables: /content/CPDIF_Results/Tables
Models: /content/CPDIF_Results/Models

CPDIF-S-LR test-set ITE mean (percentage points): 0.1312276376037168
CPDIF-S-LR persuadable fraction: 0.1284156914360926
Primary downstream model: CPDIF-S-LR
Best validation benchmark configuration: CPDIF-S-LR

Review Experiment_Configuration.json and Run_Summary.json.
